# Milestone 2

The primary objective of Milestone 2 is to transition from classical NLP techniques (like TF-IDF and bag-of-words) to modern, state-of-the-art deep learning architectures.

Instead of relying solely on surface-level word overlap (lexical similarity), now we focuse on:

Getting hands-on with the Hugging Face ecosystem (transformers and datasets libraries).

Understanding Attention Mechanisms and how models like BERT/RoBERTa generate context-aware representations.

Leveraging dense sentence embeddings (sentence-transformers) to significantly improve candidate ranking over basic TF-IDF baselines.

Exploring advanced paradigms such as Zero-Shot Classification and Generative QA using Small Language Models (SLMs).


## W & B setup

In [ ]:
# import wandb
# wandb.login(key="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")


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [ ]:
# wandb.init(
#     project="24f2007883-dl-genai-project",
#     name="milestone-2-transformer-baseline"
# )

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


# Introduction to Hugging Face transformers & datasets
Focus: Data loading, tokenizer properties, and tensor representation.

## Question 1
"Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here."

In [17]:
## Import Libraries 
## Load Dataset using Hugging Face Datasets
from datasets import load_dataset 
dataset = load_dataset(
    "csv",
    data_files="../data/train.csv"
)
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
        num_rows: 2000
    })
})


### Combined Text Function

In [ ]:
train_dataset = dataset["train"]

# Function to create new column of combine text 
def combine_text(example):
    example["combined_text"] = (
        example["prompt"] + " " + example["A"]
    )
    return example

train_dataset = train_dataset.map(combine_text)
print(train_dataset[0]["combined_text"])
print()
length_at_51 = len(train_dataset[51]["combined_text"])
print(f"length of character at 51th index : {length_at_51}")

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.

length of character at 51th index : 614


## Question 2 : Initialize the bert-base-uncased tokenizer

"Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?" 

In [ ]:
from transformers import AutoTokenizer 

tokenizer = AutoTokenizer.from_pretrained(       # it inspects the model's configuration file,  
    "bert-base-uncased"                          # detects which architecture was used,
)                                                # and automatically loads the matching tokenizer class and vocabulary for us.

# how tokenizer and model works : 
# Raw Text ──> AutoTokenizer ──> Numbers (Token IDs) ──> AutoModel ──> Predictions / Output

print(tokenizer.vocab_size)
print(tokenizer.convert_tokens_to_ids("[SEP]"))

30522
102


## Question 3 : Extract the [SEP] token ID
"Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token."

In [ ]:
print(tokenizer.cls_token, tokenizer.cls_token_id)        # Pura sentence/text ka overall summary (meaning) collect karna. it present at the starting.
print(tokenizer.sep_token, tokenizer.sep_token_id)        # Sentence boundaries aur multi-sentence input ko clean tarike se differentiate karna.
print(tokenizer.pad_token, tokenizer.pad_token_id)
print(tokenizer.unk_token, tokenizer.unk_token_id)
print(tokenizer.mask_token, tokenizer.mask_token_id)

[CLS] 101
[SEP] 102
[PAD] 0
[UNK] 100
[MASK] 103


# Question 4 : Tokenization

"Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?"

In [ ]:
import torch 
prompts = train_dataset["prompt"][:]

try:
    encoding = tokenizer(
        prompts,                    # Tokenizer har text ko individual word/sub-word tokens me break karta hai.
        padding="max_length",       # Chote sentences ke peeche [PAD] tokens (ID: 0) add kar deta hai.
        truncation=True,            # Agar koi prompt bohot lamba hai (jaise 200 words ka), toh max length ke baad wale baaki words ko cut (truncate) kar deta hai.
        max_length=128,             # Sequence ki maximum target length 128 tokens set karta hai. 128 tak push hone ke liye [PAD] add honge.
        return_tensors="pt"         # pt = torch
    )
except Exception as e:
    print(e)

print(encoding["input_ids"].shape) # keys in tokenizer : dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

torch.Size([2000, 128])



# Question 6: BERT/RoBERTa Architecture
"Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

(Note: We follow zero-indexing here.)"

In [ ]:
# head dimension = hidden size / no. of heads

# Load model 
from transformers import AutoModel            # AutoModel ka kaam: (Numbers $\rightarrow$ Embeddings/Vectors)

model = AutoModel.from_pretrained("bert-base-uncased")

text = train_dataset[0]["prompt"]

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)     # $$\text{Shape} = (\text{Batch Size}, \text{Sequence Length}, \text{Hidden Size})$$

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2840.86it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


sequence length = 31  , sequence_length (e.g., 216): Prompt ke andar kitne total tokens (input_ids) the.

# Question 7: 
"Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places)."

In [23]:
cls_vector = outputs.last_hidden_state[0, 0]
print(cls_vector.shape)
print(cls_vector[:5])
print(round(cls_vector[:5].sum().item(), 4))

torch.Size([768])
tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
-1.2001


# Question 8:
"Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)?? (Round your answer to 4 decimal places)."

In [ ]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(
    text,
    return_tensors="pt"
)
with torch.no_grad():
    outputs = model(**inputs)

print(len(outputs.attentions))             # Attention Matrix Size = (Sequence Length x Sequence Length)
print(outputs.attentions[-1].shape)        # Attention Matrix ,sequence ke har ek token ka baaki saare tokens ke saath kitna relation/connection hai, uski ek mathematical grid hai.
head0 = outputs.attentions[-1][0, 0]       # Attention Weight: Is relation ki strength ko $0.0$ se $1.0$ ke beech ek probability/weight se show kiya jata hai.
print(head0.shape)  

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3182.39it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


12
torch.Size([1, 12, 10, 10])
torch.Size([10, 10])


In [25]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for i, token in enumerate(tokens):
    print(i, token)

attention_value = head0[0, 4]
print(attention_value.item())
print(round(attention_value.item(), 4))

0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]
0.10247300565242767
0.1025


##### 📌 Understanding Transformer Attention Tensor Indexing

> **Tensor Example:** `outputs.attentions[-1][0, 0][0, 4]`

---

##### 🔍 Breakdown of Indices
* **`[-1]` (Last Layer):** 12th layer ka attention tensor pick kiya.
* **`[0]` (Batch Index 0):** Pehle (aur akle) sentence/prompt ki file pick ki.
* **`[0]` (Head Index 0):** Total 12 attention heads mein se **pehle head ki $10 \times 10$ matrix grid** uthai.
* **`[0, 4]` (Row 0, Column 4):** Row 0 (`[CLS]` token) se Column 4 (`fusion` token) ka **attention weight** read kiya.

---

##### 💡 Core Takeaway
* **Sequence Length = 10 Tokens:** Isliye Matrix Size $10 \times 10$ hai.
* **12 Attention Heads:** Har layer $10 \times 10$ ki total **12 grids** banati hai, jisme se humne Head 0 ki grid select ki.

# Question 9: Context-Aware Embeddings

"Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here." 

In [26]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import util

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

prompt = train_dataset[0]["prompt"]
option_b = train_dataset[0]["B"]

prompt_embedding = model.encode(
    prompt,
    convert_to_tensor=True
)

option_embedding = model.encode(
    option_b,
    convert_to_tensor=True
)

score = util.cos_sim(
    prompt_embedding,
    option_embedding
)

print(score)
print(round(score.item(),4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3568.74it/s]


tensor([[0.7658]])
0.7658


In [27]:
print(prompt_embedding.shape)
print(option_embedding.shape)

torch.Size([384])
torch.Size([384])


Sentence-Transformer (Sentence-level Embedding):

all-MiniLM-L6-v2 ek specialized lightweight model hai. Yeh pure sentence ke sabhi tokens par Mean Pooling apply karke sentence ka ek single vector representation bana deta hai.

torch.Size([384]) Ka Practical Meaning:
Model ne poore prompt sentence ke overall meaning, context, aur key concepts ko condense (compress) karke 384 numbers ki list bana di.

# Question 10: Pipelines
"Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?"

In [ ]:
# pipeline 1 :
tfidf_map3 = 0.2961666666666667 

# Pipeline 2 :
from sentence_transformers import SentenceTransformer, util

# Load MiniLM model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def map_at_3(actual, prediction) :
    if actual in prediction :
        rank = prediction.index(actual) + 1 

        return 1 / rank 
    return 0

scores = []
minilm_top3_predictions = [] 

for i, row in enumerate(train_dataset):

    if i % 100 == 0:
        print(f"Processing {i}...")

    prompt = row["prompt"]
    actual = row["answer"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    # Prompt embedding
    prompt_embedding = model.encode(
        prompt,
        convert_to_tensor=True
    )

    # Similarity scores
    similarities = {}

    for label, text in options.items():

        option_embedding = model.encode(
            text,
            convert_to_tensor=True
        )

        similarities[label] = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()

    # Rank options
    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]
    minilm_top3_predictions.append(top3)

    scores.append(
        map_at_3(actual, top3)
    )

# Final MAP@3
final_map3 = sum(scores) / len(scores)

print("MiniLM MAP@3 :", final_map3)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1409.79it/s]


Processing 0...
Processing 100...
Processing 200...
Processing 300...
Processing 400...
Processing 500...
Processing 600...
Processing 700...
Processing 800...
Processing 900...
Processing 1000...
Processing 1100...
Processing 1200...
Processing 1300...
Processing 1400...
Processing 1500...
Processing 1600...
Processing 1700...
Processing 1800...
Processing 1900...
MiniLM MAP@3 : 0.4230833333333333


In [ ]:
# improvement count

import pickle

# Load TF-IDF predictions
with open("tfidf_top3_predictions.pkl", "rb") as f:
    tfidf_top3_predictions = pickle.load(f)

improved_count = 0

for i, row in enumerate(train_dataset):

    actual = row["answer"]

    tfidf_correct = actual in tfidf_top3_predictions[i]
    minilm_correct = actual in minilm_top3_predictions[i]

    if (not tfidf_correct) and minilm_correct:
        improved_count += 1

print("Improved Count =", improved_count)

Improved Count = 502


### 📌 Key Takeaway: TF-IDF vs. Dense Embeddings (MiniLM)

> **Core Finding:** While both models achieved a similar overall MAP@3 score (~0.42), their underlying search mechanisms operate fundamentally differently.

---

##### 🔍 Key Insights
* **Lexical vs. Semantic Search:** TF-IDF relies strictly on exact keyword matching, whereas `all-MiniLM-L6-v2` captures semantic context and meaning.
* **The 502 Improved Questions:** MiniLM successfully correctly ranked the answer in the Top-3 for **502 questions** where TF-IDF completely failed due to a lack of overlapping keywords.
* **Conclusion:** Dense context-aware embeddings handle paraphrasing and conceptual relevance far better than classical frequency-based NLP techniques.

# Question 11: Zero-Shot Classification
"Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places)."

In [34]:
from transformers import pipeline

# Zero-shot classification pipeline
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Row index 1 (2nd row)
sequence = train_dataset[1]["prompt"]

candidate_labels = [
    train_dataset[1]["A"],
    train_dataset[1]["B"],
    train_dataset[1]["C"]
]

# Prediction
result = classifier(
    sequence,
    candidate_labels=candidate_labels
)

print(result)

print("\nTop Label :", result["labels"][0])
print("Top Score :", round(result["scores"][0], 4))

e:\Projects\GenAi\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ps928\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2208.96it/s]


{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

### 📌 Understanding Zero-Shot Classification (`facebook/bart-large-mnli`)

> **Key Idea:** Zero-shot classification categorizes text into user-defined classes without any model fine-tuning.

---

##### 🔍 Parameter & Structure Breakdown

1. **`candidate_labels`:** 
   * Passed as possible categories/options (e.g., Options A, B, C).
   * The pipeline treats each option as a natural language hypothesis and evaluates how strongly the prompt implies it.

2. **Output Dictionary Structure (`result`):**
   * **`result['labels']`:** Candidate options sorted in descending order by model confidence.
   * **`result['labels'][0]`:** The top-ranked winning option.
   * **`result['scores'][0]`:** The highest probability score associated with the top-ranked option (uses **Softmax** by default, summing total probabilities across candidates to 1.0).

# Question 12:
"Repeat the zero-shot classification evaluation for the same 2nd row prompt and same candidate labels (Options A, B, C), but set multi_label=True.

Calculate the sum of all probability scores obtained with multi_label=True and subtract it from the sum of all probability scores obtained with multi_label=False (from Question 11).

What is the absolute difference between the two sums, rounded to 4 decimal places?"

In [36]:
# Softmax (already done)
result_softmax = classifier(
    sequence,
    candidate_labels=candidate_labels
)

# Multi-label (Sigmoid)
result_sigmoid = classifier(
    sequence,
    candidate_labels=candidate_labels,
    multi_label=True
)

print("Softmax Scores :", result_softmax["scores"])
print("Sigmoid Scores :", result_sigmoid["scores"])

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])
print(result_sigmoid)
difference = abs(softmax_sum - sigmoid_sum)

print("\nSoftmax Sum :", softmax_sum)
print("Sigmoid Sum :", sigmoid_sum)

print("\nAbsolute Difference :", round(difference, 4))

Softmax Scores : [0.4574529826641083, 0.27506425976753235, 0.267482727766037]
Sigmoid Scores : [0.00046927170478738844, 2.0635867258533835e-05, 1.9700542907230556e-05]
{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between t

### 📌 Key Takeaway: Softmax vs. Sigmoid (`multi_label`)

> **Core Difference:** `multi_label=False` uses **Softmax** across candidates, forcing total probabilities to sum to 1.0. `multi_label=True` uses independent **Sigmoids**, evaluating each candidate separately.

---

##### 🔍 Key Insights
* **Softmax (`multi_label=False`):** Assumes single-label classification where options compete. The sum of all option scores is always **1.0**.
* **Sigmoid (`multi_label=True`):** Treats each candidate independently (range $0.0$ to $1.0$). Sum of probabilities can be significantly less than or greater than 1.0 depending on independent confidence.
* **Resulting Difference:** |1.0 - 0.0005| = 0.9995.

# Question 13 (Generative AI with SLM):
"Initialize the text2text-generation pipeline using google/flan-t5-small. Pass a prompt formatted like: "Answer the following question by selecting the best option: {prompt} Options: A) {A}, B) {B}. Answer:" for row ID 0. What is the model's generated response (the exact output string)?"

In [39]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = train_dataset[0]["prompt"]
option_a = train_dataset[0]["A"]
option_b = train_dataset[0]["B"]

query = (
    f"Question: {prompt}. "
    f"Is the correct answer A: {option_a} "
    f"or B: {option_b}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer(query, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2786.37it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


B


In [38]:
import transformers
print(transformers.__version__)

5.12.1


In [ ]:
# wandb.log({
#     "train_loss": train_loss,
#     "val_loss": val_loss,
#     "map3": map3
# })